In [1]:
import torch
import transformers
import huggingface_hub
import sentence_transformers
import ipywidgets

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch: 2.14.0
Transformers: 4.57.6
Hugging Face Hub: 0.36.2
Sentence Transformers: 5.7.0
GPU available: False


In [3]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

In [4]:
import json, re, time, math, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-0.6B"
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print("Device:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE != "cuda":
    model.to(DEVICE)
model.eval()
print("Ready")

Device: mps
Ready


In [6]:
import torch

def llm(messages, max_new_tokens=160):
    """
    Run local LLM inference using Hugging Face Transformers.

    Features:
    - No paid API
    - No API key
    - Supports system, user, and assistant messages
    - Disables Qwen3 thinking mode
    - Uses deterministic greedy decoding
    - Explicitly supplies attention mask
    """

    # --------------------------------------------------
    # 1. Convert chat messages into model input
    # --------------------------------------------------

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt"
    )

    # Move input tensors to the model's device
    inputs = {
        key: value.to(next(model.parameters()).device)
        for key, value in inputs.items()
    }

    # --------------------------------------------------
    # 2. Generate response
    # --------------------------------------------------

    with torch.inference_mode():

        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],

            max_new_tokens=max_new_tokens,

            # Deterministic generation
            do_sample=False,

            # Explicitly disable sampling parameters
            temperature=None,
            top_p=None,
            top_k=None,

            pad_token_id=tokenizer.eos_token_id
        )

    # --------------------------------------------------
    # 3. Decode only newly generated tokens
    # --------------------------------------------------

    input_length = inputs["input_ids"].shape[-1]

    generated_tokens = outputs[0][input_length:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()


def ask(
    question,
    system="You are a concise, helpful assistant."
):
    """
    Simple interface for asking questions.
    """

    messages = [
        {
            "role": "system",
            "content": system
        },
        {
            "role": "user",
            "content": question
        }
    ]

    return llm(messages)


# --------------------------------------------------
# TEST
# --------------------------------------------------

print(
    ask(
        "In one sentence, what is function calling?"
    )
)

Function calling is the process of invoking a function or method in a program to achieve a specific task.


In [7]:
print(ask('Who are you?'))

I am a helpful assistant. What can I do for you?


In [8]:
print(ask('Who is Krishna?'))

Krishna is the divine, spiritual leader and the son of Vishnu. He is revered as the embodiment of love, compassion, and divine wisdom. He is often depicted as a compassionate and enlightened being, guiding and protecting the devotees.


In [9]:
print(ask('he is not son of vishnu'))

He is not the son of Vishnu.


In [10]:
print(ask('Who is swathi?'))

Swathi is a fictional character from the Indian television series *Dilwale Dulhania Kapoor**. She is a young woman who plays the role of a journalist and a writer.


In [11]:
print(ask('python code to sum two numbers'))

```python
# Example code to sum two numbers
num1 = 10
num2 = 20
sum = num1 + num2
print("Sum of the two numbers:", sum)
```


In [12]:
# Fictional policy and deterministic tools
ATTENDANCE_THRESHOLD = 75.0

def attendance(attended: int, total: int):
    if type(attended) is not int or type(total) is not int:
        return {"error": "attended and total must be integers"}
    if total <= 0 or attended < 0 or attended > total:
        return {"error": "invalid attendance values"}
    pct = round(100 * attended / total, 2)
    return {"percentage": pct, "eligible": pct >= ATTENDANCE_THRESHOLD,
            "required_percentage": ATTENDANCE_THRESHOLD}

def calculator(a: float, b: float, operation: str):
    if operation not in {"add", "subtract", "multiply", "divide"}:
        return {"error": "unsupported operation"}
    if operation == "divide" and b == 0:
        return {"error": "division by zero"}
    return {"result": {"add":lambda:a+b,"subtract":lambda:a-b,
                       "multiply":lambda:a*b,"divide":lambda:a/b}[operation]()}

TOOLS = {"attendance": attendance, "calculator": calculator}
print(attendance(32,40), calculator(34567,98765,"multiply"))

{'percentage': 80.0, 'eligible': True, 'required_percentage': 75.0} {'result': 3414009755}


In [14]:
TOOL_ROUTER_SYSTEM = """You select tools for a university assistant.
Return ONE JSON object only, without markdown or explanation.
Allowed formats:
{"tool":"attendance","arguments":{"attended":32,"total":40}}
{"tool":"calculator","arguments":{"a":12,"b":3,"operation":"multiply"}}
{"tool":"none","arguments":{}}
Only choose attendance when both attended and total are explicitly given.
Only choose calculator for an explicit arithmetic operation with two operands.
If values are missing, select none. Never invent arguments.
"""

def parse_json_object(text):
    """Parse a complete JSON object; no eval(), no code execution."""
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        return None
    return obj if isinstance(obj, dict) else None

def execute_tool_request(obj):
    if not isinstance(obj, dict) or set(obj) != {"tool", "arguments"}:
        return {"error": "invalid tool request schema"}
    name, args = obj["tool"], obj["arguments"]
    if name == "none":
        return {"status": "no tool selected"} if args == {} else {"error":"unexpected arguments"}
    if name not in TOOLS or not isinstance(args, dict):
        return {"error": "unknown tool or invalid arguments"}
    if name == "attendance":
        if set(args) != {"attended", "total"}:
            return {"error":"attendance requires attended and total"}
        return attendance(**args)
    if set(args) != {"a", "b", "operation"}:
        return {"error":"calculator arguments mismatch"}
    if type(args["a"]) not in (int,float) or type(args["b"]) not in (int,float) or type(args["operation"]) is not str:
        return {"error":"invalid calculator argument types"}
    if not all(math.isfinite(v) for v in (args["a"], args["b"])):
        return {"error":"non-finite input"}
    return calculator(**args)

def call_tool(question):
    raw = llm([{"role":"system","content":TOOL_ROUTER_SYSTEM},
               {"role":"user","content":question}], max_new_tokens=120)
    obj = parse_json_object(raw)
    result = execute_tool_request(obj)
    return {"model_output":raw, "parsed":obj, "tool_result":result}

for question in ["I attended 32 out of 40 classes. Am I eligible?",
                 "Multiply 34567 by 98765", "What is my attendance?"]:
    print("\nQUESTION:",question,"\n",call_tool(question))


QUESTION: I attended 32 out of 40 classes. Am I eligible? 
 {'model_output': '{"tool":"attendance","arguments":{"attended":32,"total":40}}', 'parsed': {'tool': 'attendance', 'arguments': {'attended': 32, 'total': 40}}, 'tool_result': {'percentage': 80.0, 'eligible': True, 'required_percentage': 75.0}}

QUESTION: Multiply 34567 by 98765 
 {'model_output': '{"tool":"calculator","arguments":{"a":34567,"b":98765,"operation":"multiply"}}', 'parsed': {'tool': 'calculator', 'arguments': {'a': 34567, 'b': 98765, 'operation': 'multiply'}}, 'tool_result': {'result': 3414009755}}

QUESTION: What is my attendance? 
 {'model_output': '{"tool":"attendance","arguments":{"attended":32,"total":40}}', 'parsed': {'tool': 'attendance', 'arguments': {'attended': 32, 'total': 40}}, 'tool_result': {'percentage': 80.0, 'eligible': True, 'required_percentage': 75.0}}
